In [135]:
import pandas as pd
import ast

In [136]:
# Combine and process all relevant data into one DataFrame
df_rental = pd.read_csv(r"rea_data\domain\Data\vic_rentals_all_cleaned.csv")
df_bus = pd.read_csv(r"rentals_distances_to_bus_stops.csv")
df_bus.rename(columns={"rental_id":"listing_id", "top_distances":"top_bus_distances"}, inplace=True)
df_train_cbd = pd.read_csv(r"rentals_distances_to_train_stops_and_cbd.csv")
df_train_cbd.rename(columns={"rental_id":"listing_id", "top_distances":"top_train_distances"}, inplace=True)
df_tram = pd.read_csv(r"rentals_distances_to_tram_stops.csv")
df_tram.rename(columns={"rental_id":"listing_id", "top_distances":"top_tram_distances"}, inplace=True)
df_school = pd.read_csv(r"rentals_distances_to_schools.csv")
df_school.rename(columns={"rental_id":"listing_id", "top_distances":"top_school_distances"}, inplace=True)

df = df_rental.merge(df_bus, on="listing_id", how="left")
df = df.merge(df_train_cbd, on="listing_id", how="left")
df = df.merge(df_tram, on="listing_id", how="left")
df = df.merge(df_school, on="listing_id", how="left")

df["top_bus_distances"] = df["top_bus_distances"].str.replace("inf", "None")

for col in ["top_bus_distances", "top_train_distances", "top_tram_distances", "top_school_distances", 
            "top_bus_ids", "top_station_ids", "top_tram_ids", "top_school_ids"]:
    df[col] = df[col].apply(lambda x:ast.literal_eval(x) if pd.notnull(x) else [])

df["closest_bus_distance"] = df["top_bus_distances"].str[0]
df["closest_train_distance"] = df["top_train_distances"].str[0]
df["closest_tram_distance"] = df["top_tram_distances"].str[0]
df["closest_school_distance"] = df["top_school_distances"].str[0]
df["closest_bus_id"] = df["top_bus_ids"].str[0]
df["closest_train_id"] = df["top_station_ids"].str[0]
df["closest_tram_id"] = df["top_tram_ids"].str[0]
df["closest_school_id"] = df["top_school_ids"].str[0]


In [137]:
df.head()

,listing_id,suburb,postcode,weekly_rent,bond,available_date,date_listed,days_listed,bedrooms,bathrooms,...,top_school_distances,top_school_ids,closest_bus_distance,closest_train_distance,closest_tram_distance,closest_school_distance,closest_bus_id,closest_train_id,closest_tram_id,closest_school_id
0,16782629,south kingsville,3015,460.0,1994.0,"Tuesday, 02 September 2025",2025-08-13,27.0,2.0,1.0,...,"[1060.45, 2564.66, 4788.34, 115324.43, 224471.84]","[77, 4788, 1527, 3659, 113]",769.19,1889.09,5684.28,1060.45,40836,15345,17934.0,77.0
1,17471867,south kingsville,3015,400.0,1738.0,"Thursday, 27 March 2025",2025-03-06,187.0,2.0,1.0,...,"[4190.38, 39213.27, 41875.79, 43234.16, 46083.64]","[3988, 4788, 113, 3659, 1527]",516.57,2230.27,6025.46,4190.38,40834,15345,17934.0,3988.0
2,17721851,south kingsville,3015,795.0,3454.0,"Monday, 15 September 2025",2025-08-19,21.0,3.0,2.0,...,"[1777.42, 23266.69, 23738.09, 24221.5, 25347.84]","[3659, 1527, 1690, 4788, 113]",17496.04,3156.90,6073.35,1777.42,40820,vic:rail:SPT,17934.0,3659.0
3,17725855,south kingsville,3015,675.0,2933.0,"Wednesday, 20 August 2025",2025-08-21,19.0,3.0,1.0,...,"[38929.89, 41592.41, 42950.78, 43634.11, 45800...","[4788, 113, 3659, 5278, 1527]",2844.32,1946.90,5742.08,38929.89,9006,15345,17934.0,4788.0
4,17745057,south kingsville,3015,450.0,1955.0,"Tuesday, 02 September 2025",2025-09-03,6.0,2.0,1.0,...,"[4109.06, 39131.94, 41794.46, 43152.83, 46002.31]","[3988, 4788, 113, 3659, 1527]",435.24,2148.95,5944.13,4109.06,40834,15345,17934.0,3988.0


In [138]:
# Group data by suburb
df_suburb = df.groupby('suburb').agg({
  'closest_bus_distance' :'mean',
  'closest_train_distance' :'mean',
  'closest_tram_distance' :'mean',
  'closest_school_distance' :'mean'
})

In [139]:
df_suburb = df_suburb.reset_index()
df_suburb

,suburb,closest_bus_distance,closest_train_distance,closest_tram_distance,closest_school_distance
0,abbotsford,14604.013000,3652.688333,1373.042333,1033.434000
1,aberfeldie,2222.946000,1589.060000,11626.160000,11679.786000
2,airport west,4263.098000,6269.650000,1200.716400,2626.631200
3,albanvale,7527.106000,4563.546000,12915.138000,5580.066000
4,albert park,7052.681667,18515.161667,604.373333,1730.943333
...,...,...,...,...,...
636,yarrambat,37244.320000,19925.740000,8498.520000,22518.850000
637,yarraville,7170.406552,1568.921724,3251.017931,1483.590345
638,yarrawonga,196103.697333,249847.898667,265162.486000,48936.769333
639,yarroweyah,192374.090000,223618.830000,238503.640000,8729.440000


In [140]:
df_crime = pd.read_csv(r"crime_per_suburb.csv")
df_crime.rename(columns={"Suburb/Town Name":"suburb"}, inplace=True)
df_suburb = df_suburb.merge(df_crime, on="suburb", how="left")

df_hospital = pd.read_csv(r"hospital_per_suburb.csv")
df_hospital.rename(columns={"locality_name":"suburb"}, inplace=True)
df_suburb = df_suburb.merge(df_hospital, on="suburb", how="left")

df_suburb['Offence Count']=df_suburb['Offence Count'].fillna(0)
df_suburb['num_hospitals']=df_suburb['num_hospitals'].fillna(0)

In [141]:
df_suburb

,suburb,closest_bus_distance,closest_train_distance,closest_tram_distance,closest_school_distance,Offence Count,num_hospitals
0,abbotsford,14604.013000,3652.688333,1373.042333,1033.434000,14791.0,0.0
1,aberfeldie,2222.946000,1589.060000,11626.160000,11679.786000,1429.0,0.0
2,airport west,4263.098000,6269.650000,1200.716400,2626.631200,9294.0,0.0
3,albanvale,7527.106000,4563.546000,12915.138000,5580.066000,3766.0,0.0
4,albert park,7052.681667,18515.161667,604.373333,1730.943333,6087.0,0.0
...,...,...,...,...,...,...,...
636,yarrambat,37244.320000,19925.740000,8498.520000,22518.850000,525.0,0.0
637,yarraville,7170.406552,1568.921724,3251.017931,1483.590345,9435.0,0.0
638,yarrawonga,196103.697333,249847.898667,265162.486000,48936.769333,4825.0,1.0
639,yarroweyah,192374.090000,223618.830000,238503.640000,8729.440000,255.0,0.0


In [142]:
from sklearn.preprocessing import MinMaxScaler

# Calculate education score, infrastructure score to compute livability score per suburb
df_suburb['mean_transport_distance'] = df_suburb[['closest_train_distance', 'closest_tram_distance',
                                                  'closest_bus_distance']].mean(axis=1)
scaler = MinMaxScaler()

df_suburb['education_score'] = 1 - scaler.fit_transform(df_suburb[['closest_school_distance']])
df_suburb['infrastructure_score'] = 1 - scaler.fit_transform(df_suburb[['mean_transport_distance']])
df_suburb['stability_score'] = 1 - scaler.fit_transform(df_suburb[['Offence Count']])
df_suburb['healthcare_score'] = scaler.fit_transform(df_suburb[['num_hospitals']])

df_suburb['education_score']=df_suburb['education_score'].fillna(0)
df_suburb['infrastructure_score']=df_suburb['infrastructure_score'].fillna(0)
df_suburb['stability_score']=df_suburb['stability_score'].fillna(0)
df_suburb['healthcare_score']=df_suburb['healthcare_score'].fillna(0)

suburb_score = df_suburb[['suburb','education_score', 'infrastructure_score', 'stability_score', 'healthcare_score']].copy()
suburb_score['livability_score'] = (suburb_score['education_score'] + suburb_score['infrastructure_score'] +
                                    suburb_score['stability_score'] + suburb_score['healthcare_score'])
suburb_score.sort_values('livability_score', ascending=False, inplace=True)


In [143]:
suburb_score



,suburb,education_score,infrastructure_score,stability_score,healthcare_score,livability_score
195,east melbourne,9.868722e-01,0.992104,0.954332,1.0,3.933309
219,footscray,9.990109e-01,0.995688,0.858995,0.8,3.653693
457,parkville,9.977456e-01,0.988080,0.962574,0.7,3.648399
136,clayton,9.966136e-01,0.991944,0.914723,0.7,3.603280
609,werribee,9.958396e-01,0.975511,0.782573,0.7,3.453923
...,...,...,...,...,...,...
412,nelson,3.417113e-01,0.241132,0.999372,0.0,1.582216
491,robinvale,1.515303e-01,0.166856,0.983918,0.1,1.402304
286,irymple,2.093790e-01,0.033550,0.984391,0.0,1.227320
115,cardross,2.220446e-16,0.038053,0.998485,0.0,1.036539
